In [1]:
import pandas as pd
import numpy as np

def ovr_ll(probs, y):
    p = np.clip(probs, 1e-15, 1 - 1e-15)
    oh = np.eye(3)[y.astype(int)]
    return (oh * -np.log(p) + (1 - oh) * -np.log(1 - p)).mean()

BASE = r"C:\Users\ruben\Downloads\Contents260916"

for ds in ["full", "lite", "lite_recortado"]:
    ruta = f"{BASE}\\stage2_mlp_predictions_{ds}.csv"
    try:
        df = pd.read_csv(ruta, encoding="latin-1")
    except UnicodeDecodeError:
        df = pd.read_csv(ruta, encoding="utf-8-sig")

    g = df.groupby("match_id_new").agg(
        y=("y_true","first"),
        gH=("glicko_prob_H","mean"), gD=("glicko_prob_D","mean"), gA=("glicko_prob_A","mean"),
        bH=("mlp_base_prob_H","mean"), bD=("mlp_base_prob_D","mean"), bA=("mlp_base_prob_A","mean"),
        fH=("mlp_full_prob_H","mean"), fD=("mlp_full_prob_D","mean"), fA=("mlp_full_prob_A","mean"),
    )
    y = g["y"].values

    print(f"\n=== {ds} ===")
    print(f"  Glicko aislado : {ovr_ll(g[['gH','gD','gA']].values, y):.4f}")
    print(f"  MLP base       : {ovr_ll(g[['bH','bD','bA']].values, y):.4f}")
    print(f"  MLP completo   : {ovr_ll(g[['fH','fD','fA']].values, y):.4f}")


=== full ===
  Glicko aislado : 0.6064
  MLP base       : 0.5975
  MLP completo   : 0.5978

=== lite ===
  Glicko aislado : 0.6058
  MLP base       : 0.5964
  MLP completo   : 0.5959

=== lite_recortado ===
  Glicko aislado : 0.6064
  MLP base       : 0.5975
  MLP completo   : 0.5989
